# 03 — MAGE Augmentation Framework (Paper 2)

Reproduce the MAGE (Multi-Head Attention Guided Embeddings) experiments from arXiv:2502.17987.

**Pipeline**:
1. Extract sentence embeddings from AfriBERTa
2. Apply LiDA (Language-Independent Data Augmentation)
3. Compress/reconstruct via DAE or VAE
4. Classify with MAGE multi-head attention classifier

**Dataset**: AfriSenti-SemEval (Kinyarwanda, Swahili, Xitsonga)

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mrm.augmentation.lida import LiDA
from mrm.models.autoencoder import DenoisingAutoencoder, VariationalAutoencoder, train_autoencoder
from mrm.models.mage import MAGEClassifier, LSTMClassifier, LogisticRegressionWrapper
from mrm.data.datasets import EmbeddingDataset
from mrm.evaluation.metrics import weighted_f1, accuracy_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEVICE}')

## 1. Extract AfriBERTa Embeddings

In [ ]:
from mrm.models.transformer_clf import load_transformer_classifier
from mrm.data.datasets import HFTextDataset

LANGUAGE = 'kin'  # change to 'swa' or 'tso'

_, tokenizer = load_transformer_classifier('afriberta', num_labels=3)
emb_model, _ = load_transformer_classifier('afriberta', num_labels=3)
emb_model = emb_model.base_model.to(DEVICE)

train_ds = HFTextDataset.from_csv(f'../data/processed/afrisenti/{LANGUAGE}_train.csv', tokenizer)
test_ds  = HFTextDataset.from_csv(f'../data/processed/afrisenti/{LANGUAGE}_test.csv', tokenizer)
print(f'AfriSenti [{LANGUAGE}]: {len(train_ds)} train, {len(test_ds)} test')

## 2. LiDA Augmentation

In [ ]:
lida = LiDA(r_min=0.0, r_max=0.1)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32)
test_loader  = torch.utils.data.DataLoader(test_ds, batch_size=32)

train_emb, train_labels = lida.augment_dataset(emb_model, train_loader, device=DEVICE, n_augmented=2)
test_emb,  test_labels  = lida.augment_dataset(emb_model, test_loader,  device=DEVICE, n_augmented=0)

print(f'Original: {len(train_ds)} → Augmented: {len(train_emb)} embeddings (3× via LiDA)')
print(f'Embedding shape: {train_emb.shape}')

## 3. Train DAE and VAE

In [ ]:
dae = DenoisingAutoencoder(input_dim=768, latent_dim=32)
dae_losses = train_autoencoder(dae, train_emb, num_epochs=30, lr=1e-3, device=DEVICE)

vae = VariationalAutoencoder(input_dim=768, hidden_dim=512, latent_dim=256)
vae_losses = train_autoencoder(vae, train_emb, num_epochs=30, lr=1e-3, device=DEVICE, is_vae=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(dae_losses, label='DAE')
ax.plot(vae_losses, label='VAE')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Autoencoder Training Loss')
ax.legend()
plt.tight_layout()
plt.show()

## 4. MAGE Classification and Ablation

In [ ]:
def run_mage_variant(ae, is_vae, train_emb, train_labels, test_emb, test_labels, n_epochs=20):
    ae.to(DEVICE).eval()
    with torch.no_grad():
        if is_vae:
            ref_train, _, _ = ae(train_emb.to(DEVICE))
            ref_test, _, _  = ae(test_emb.to(DEVICE))
        else:
            ref_train = ae(train_emb.to(DEVICE))
            ref_test  = ae(test_emb.to(DEVICE))

    model = MAGEClassifier(embed_dim=768, num_heads=4, num_classes=3).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=2e-4)
    crit  = torch.nn.CrossEntropyLoss()
    loader = torch.utils.data.DataLoader(
        EmbeddingDataset(ref_train.cpu(), train_labels), batch_size=32, shuffle=True
    )
    for _ in range(n_epochs):
        model.train()
        for embs, lbls in loader:
            opt.zero_grad()
            crit(model(embs.to(DEVICE)), lbls.to(DEVICE)).backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        preds = model(ref_test).argmax(1).cpu().tolist()
    return weighted_f1(test_labels.tolist(), preds)

f1_dae = run_mage_variant(dae, False, train_emb, train_labels, test_emb, test_labels)
f1_vae = run_mage_variant(vae, True,  train_emb, train_labels, test_emb, test_labels)

lr = LogisticRegressionWrapper()
lr.fit(train_emb, train_labels)
lr_preds = lr.predict(test_emb)
f1_lr = weighted_f1(test_labels.tolist(), lr_preds.tolist())

print(f'MAGE+DAE  F1: {f1_dae:.4f}')
print(f'MAGE+VAE  F1: {f1_vae:.4f}')
print(f'LR Base   F1: {f1_lr:.4f}')
print(f'MAGE+DAE improvement over LR: +{(f1_dae - f1_lr)*100:.2f}%')